# Multi-Modal RAG — Pipeline Demo

End-to-end walkthrough of the ingestion and query pipeline.

## Services required

| Step | Services needed |
|---|---|
| PDF ingestion | `lightonocr-vllm` (8002), `qwen35-vllm` (8000), Ollama (11434) |
| Table ingestion | Ollama only |
| Query / Agent | `gen-vllm` (8003) → **Qwen/Qwen3-32B-AWQ**, Ollama, BGE reranker (local GPU) |
| Always running | Qdrant (7333), Ollama (11434) |

## Switching between modes

Two GPU profiles — switch to free VRAM between ingestion and query.

**Switch to ingestion mode:**
```bash
docker compose -f docker/ingestion-stack/docker-compose.yaml stop gen-vllm
docker compose -f docker/ingestion-stack/docker-compose.yaml --profile ingest up -d
```

**Switch to query mode:**
```bash
docker compose -f docker/ingestion-stack/docker-compose.yaml stop lightonocr-vllm qwen35-vllm
docker compose -f docker/ingestion-stack/docker-compose.yaml --profile query up -d
```

> Excel/CSV ingestion only needs Qdrant + Ollama — no profile switch needed.

## Swapping the generation model

Edit `.env` then restart the query profile — zero code changes needed:

```
GEN_MODEL_TAG=Qwen/Qwen3-32B-AWQ          # HuggingFace model to load
GEN_SERVED_MODEL_NAME=Qwen/Qwen3-32B-AWQ  # name exposed on the API
GENERATION_MODEL=Qwen/Qwen3-32B-AWQ       # name used by the app
GEN_GPU_MEMORY_UTILIZATION=0.90           # lower for smaller models (e.g. 0.70 for 8B)
```

```bash
docker compose -f docker/ingestion-stack/docker-compose.yaml --profile query up -d --force-recreate
```

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = (Path.cwd() / '..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    OLLAMA_API_BASE,
    OLLAMA_EMBED_MODEL,
    QDRANT_URL,
    QDRANT_COLLECTION,
    OCR_API_BASE,
    OCR_MODEL,
    GENERATION_API_BASE,
    GENERATION_MODEL,
)

# ── File paths ────────────────────────────────────────────────────────────
PDF_FILE   = Path('../data/input/WHO East. Med. dsa1184.pdf')
TABLE_FILE = '../notebooks/GRC_2023_2019_11042023_103559.xlsx'

# ── Aliases for notebook cells ────────────────────────────────────────────
COLLECTION = QDRANT_COLLECTION
OLLAMA_URL = OLLAMA_API_BASE
EMBED_MODEL = OLLAMA_EMBED_MODEL
OCR_URL    = f'{OCR_API_BASE}/v1/chat/completions'
GEN_URL    = GENERATION_API_BASE
GEN_MODEL  = GENERATION_MODEL

print('Project root:', project_root)
print(f'Embed model:  {EMBED_MODEL}')
print(f'Gen model:    {GEN_MODEL}')
print(f'Qdrant:       {QDRANT_URL}  collection={COLLECTION}')

Project root: /home/karvanitis/multi-modal-rag
Embed model:  bge-m3
Gen model:    Qwen/Qwen3-32B-AWQ
Qdrant:       http://localhost:7333  collection=documents_chunks


---
## Part 1 — PDF Ingestion

**Requires ingestion profile running** (`lightonocr-vllm` on 8002, Ollama on 11434)

Pipeline:
```
PDF  →  LightOn OCR (vLLM)  →  Markdown
                                    ↓
                          Header-aware chunking
                        (split on #/##/###, max 1024 tokens)
                                    ↓
                     LLM context enrichment (optional)
                    one-sentence summary prepended to vector_text
                                    ↓
                      Ollama embedding (nomic-embed-text-v2-moe)
                            + BM42 sparse (fastembed)
                                    ↓
                        Qdrant upsert (documents_chunks)
```

### Step 1 — PDF → Markdown (LightOn OCR)

In [ ]:
from src.parser.lightonocr_parser import pdf_to_markdown

md_path = pdf_to_markdown(
    PDF_FILE,
    endpoint=OCR_URL,
    model_name=OCR_MODEL,
    markdown_tables=True,
    table_mode='grid',
)

markdown = md_path.read_text(encoding='utf-8')
print(f'Markdown saved: {md_path}')
print(f'Length: {len(markdown):,} chars | Pages: {markdown.count("<!-- PAGE ")}')  
print()
print('Preview (first 500 chars):')
print(markdown[:500])

### Step 2 — Markdown → Chunks

Chunking strategy:
- Split on `#`, `##`, `###` headers first (preserves document structure)
- Recursively split sections > 1024 tokens (100-token overlap)
- Tables (`[TABLE_START]...[TABLE_END]`) are kept intact — never split
- Tiny chunks (< 256 tokens) are merged into neighbours
- **LLM enrichment** (`enrich_with_llm=True`): each chunk gets a one-sentence summary prepended to `vector_text` (what gets embedded). Improves retrieval quality significantly.

In [ ]:
from src.chunker import chunk_markdown

chunks = chunk_markdown(
    markdown,
    enrich_with_llm=True,          # set False to skip (faster, lower quality)
    file_name=md_path.name,
    source_file=str(PDF_FILE),
)

import json
chunks_json = Path('../data/output/chunks') / f'{md_path.stem}_chunks.json'
saved = json.loads(chunks_json.read_text())

print(f'Chunks created: {len(chunks)}')
print(f'Saved to: {chunks_json}')
print()
print('Sample chunk:')
c = saved[0]
print(f'  context:   {c["context"][:120]}')
print(f'  content:   {c["content"][:200]}')
print(f'  tokens:    {c["metadata"]["token_count"]}')

### Step 3 — Embed + Upsert to Qdrant

Each chunk gets two vectors:
- **Dense** — `bge-m3` via Ollama (1024-dim, multilingual semantic similarity)
- **Sparse** — BM42 via fastembed (keyword matching, runs on CPU)

At query time, Qdrant fuses both with **RRF (Reciprocal Rank Fusion)**.

In [ ]:
from src.embedder import embed_chunks_file
from src.vector_store import ingest_embeddings

embeddings_path = embed_chunks_file(
    chunks_json,
    api_base=OLLAMA_URL,
    model_name=EMBED_MODEL,
)
print(f'Embeddings saved: {embeddings_path}')

ingest_embeddings(input_path=embeddings_path, url=QDRANT_URL, collection=COLLECTION)
print('Upserted to Qdrant.')

### (Alternative) Full pipeline in one call

In [ ]:
from src.ingest import run_ingest

result = run_ingest(
    collection=COLLECTION,
    pdf_path=PDF_FILE,
    qdrant_url=QDRANT_URL,
    ocr_endpoint=OCR_URL,
    ocr_model_name=OCR_MODEL,
    ollama_api_base=OLLAMA_URL,
    ollama_embed_model=EMBED_MODEL,
    enrich_with_llm=True,
)
print('Done:', result)

---
## Part 2 — Excel / CSV Ingestion

**Requires only Ollama** — no vLLM, no SQL, no schema extraction.

Pipeline:
```
Excel/CSV  →  load sheets (openpyxl)
                    ↓
          detect header row (heuristic)
          detect + merge units row (e.g. "(kt)", "Gg")
          extract sheet title from pre-header rows
                    ↓
          each row → text sentence:
          [File: X | Sheet: Y | <title>] | COL1: val1 | COL2: val2 | ...
                    ↓
          sheet summary chunk (1 per sheet):
          columns + unique category values
                    ↓
          Ollama embedding + BM42 sparse
                    ↓
          Qdrant upsert (same documents_chunks collection as PDFs)
```

### Step 1 — Inspect raw sheet rows

In [ ]:
import openpyxl

SHEET_NAME = 'Table1s1'

wb = openpyxl.load_workbook(TABLE_FILE, data_only=True)
ws = wb[SHEET_NAME]
raw_rows = [[cell.value for cell in row] for row in ws.iter_rows()]

print(f'Sheet: {SHEET_NAME} | Total rows: {len(raw_rows)}')
print()
print('First 6 rows (raw — includes merged title rows):')
for i, row in enumerate(raw_rows[:6]):
    non_empty = [v for v in row if v is not None and str(v).strip()]
    print(f'  [{i}] {non_empty[:5]}')

### Step 2 — Header detection + units merging

The heuristic picks the first row with ≥ 3 distinct text values (skips merged title rows).  
If the next row contains unit annotations like `(kt)`, `Gg`, `%` — they get merged into the column names.

In [ ]:
from src.ingest_table_rows import (
    _find_header_row, _detect_units_row, _merge_units_into_headers,
    _extract_sheet_title, _is_numeric
)

header_idx = _find_header_row(raw_rows)
raw_headers = [str(v).strip() if v is not None else '' for v in raw_rows[header_idx]]

units_idx = _detect_units_row(raw_rows, header_idx)
if units_idx is not None:
    headers = _merge_units_into_headers(raw_headers, raw_rows[units_idx])
    data_start = units_idx + 1
    print(f'Header row [{header_idx}]:  {raw_headers[:5]}')
    print(f'Units row  [{units_idx}]:  {[str(v).strip() for v in raw_rows[units_idx] if v][:5]}')
    print(f'Merged headers:         {headers[:5]}')
else:
    headers = raw_headers
    data_start = header_idx + 1
    print(f'Header row [{header_idx}]: {headers[:5]}')
    print('No units row detected.')

sheet_title = _extract_sheet_title(raw_rows, header_idx)
print(f'\nAuto-extracted title: "{sheet_title}"')

### Step 3 — Row → text conversion

Each row becomes one searchable sentence. The sheet title (extracted from pre-header rows) is embedded in every chunk prefix — so queries with country/year terms like *"Greece 2019"* can match.

In [ ]:
from src.ingest_table_rows import row_to_text, sheet_summary_text
from pathlib import Path

file_name = Path(TABLE_FILE).name
data_rows = raw_rows[data_start:]

# Sheet summary chunk (1 per sheet)
summary = sheet_summary_text(file_name, SHEET_NAME, headers, data_rows, sheet_title)
print('Sheet summary chunk:')
print(summary)
print()

# Row chunks (1 per data row)
sample_rows = [r for r in data_rows if any(v is not None and str(v).strip() for v in r)][:3]
print('Sample row chunks:')
for row in sample_rows:
    print(' ', row_to_text(file_name, SHEET_NAME, headers, row, sheet_title=sheet_title))
    print()

### Step 4 — Ingest full file into Qdrant

Processes all sheets, saves chunks JSON, and upserts to Qdrant.

In [ ]:
import importlib
import src.ingest_table_rows as _itr_mod
importlib.reload(_itr_mod)
from src.ingest_table_rows import ingest_table_rows

chunks_path = ingest_table_rows(TABLE_FILE, collection=COLLECTION, verbose=True)
print('\nChunks JSON saved to:', chunks_path)

### Step 5 — Verify Qdrant contents

In [ ]:
import json
from urllib.request import urlopen, Request

def qdrant_count(filter_body=None):
    url = f'{QDRANT_URL}/collections/{COLLECTION}/points/count'
    body = {"filter": filter_body} if filter_body else {}
    req = Request(url, data=json.dumps(body).encode(), method='POST',
                  headers={'Content-Type': 'application/json'})
    with urlopen(req) as r:
        return json.loads(r.read())['result']['count']

total       = qdrant_count()
table_total = qdrant_count({'must': [{'key': 'source_type', 'match': {'value': 'table'}}]})
summaries   = qdrant_count({'must': [{'key': 'metadata.chunk_type', 'match': {'value': 'sheet_summary'}}]})

print(f'Total points:     {total}')
print(f'Table chunks:     {table_total}  ({summaries} sheet summaries + {table_total - summaries} row chunks)')
print(f'Document chunks:  {total - table_total}')

---
## Bulk Ingestion & Collection Status

Ingest an entire folder at once (Excel + PDF/DOCX/PPTX) and inspect what is currently in Qdrant.

**Re-ingesting the same file is safe** — existing points are deleted before upserting, so no duplicates.

### Collection Status — what is currently ingested?

In [ ]:
import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError

def list_ingested_files(qdrant_url=QDRANT_URL, collection=QDRANT_COLLECTION):
    """Print all files currently in Qdrant with their type and chunk counts."""
    base = qdrant_url.rstrip("/")

    def _post(path, body):
        req = Request(f"{base}{path}", data=json.dumps(body).encode(),
                      headers={"Content-Type": "application/json"}, method="POST")
        with urlopen(req, timeout=30) as r:
            return json.loads(r.read())["result"]

    # Check collection exists first
    try:
        req = Request(f"{base}/collections/{collection}")
        with urlopen(req, timeout=10) as r:
            info = json.loads(r.read())
        if info.get("result", {}).get("status") != "green":
            print(f"Collection '{collection}' exists but is not ready.")
            return
    except HTTPError as e:
        if e.code == 404:
            print(f"Collection '{collection}' does not exist yet — nothing ingested.")
            return
        raise

    def _count(filter_body):
        return _post(f"/collections/{collection}/points/count", {"filter": filter_body})["count"]

    # --- Table files ---
    scroll = _post(f"/collections/{collection}/points/scroll", {
        "filter": {"must": [{"key": "source_type", "match": {"value": "table"}}]},
        "limit": 1000, "with_payload": True, "with_vector": False,
    })
    table_files: dict[str, set] = {}
    for pt in scroll["points"]:
        meta = pt["payload"].get("metadata", {})
        fname = meta.get("source_file", "unknown")
        sheet = meta.get("sheet_name", "?")
        table_files.setdefault(fname, set()).add(sheet)

    # --- Doc files ---
    scroll = _post(f"/collections/{collection}/points/scroll", {
        "filter": {"must_not": [{"key": "source_type", "match": {"value": "table"}}]},
        "limit": 2000, "with_payload": True, "with_vector": False,
    })
    doc_files: dict[str, int] = {}
    for pt in scroll["points"]:
        meta = pt["payload"].get("metadata", {})
        fname = meta.get("file_name") or meta.get("source_file", "unknown")
        doc_files[fname] = doc_files.get(fname, 0) + 1

    total_table = _count({"must": [{"key": "source_type", "match": {"value": "table"}}]})
    total_docs  = _count({"must_not": [{"key": "source_type", "match": {"value": "table"}}]})

    print(f"Collection: '{collection}'  |  total points: {total_table + total_docs}")
    print(f"  Documents : {total_docs} chunks across {len(doc_files)} file(s)")
    print(f"  Tables    : {total_table} chunks across {len(table_files)} file(s)\n")

    if doc_files:
        print("Documents:")
        for fname, n in sorted(doc_files.items()):
            print(f"   {fname:<60} {n:>5} chunks")

    if table_files:
        print("\nTables:")
        for fname, sheets in sorted(table_files.items()):
            row_count = _count({"must": [
                {"key": "source_type", "match": {"value": "table"}},
                {"key": "metadata.source_file", "match": {"value": fname}},
            ]})
            sheet_list = ', '.join(sorted(sheets)[:4]) + ('…' if len(sheets) > 4 else '')
            print(f"   {fname:<60} {row_count:>5} chunks  ({len(sheets)} sheets: {sheet_list})")

list_ingested_files()

### Wipe + Re-ingest everything from `data/input/`

Drops the current collection and ingests all files fresh.
- **Excels / CSVs** — only needs Ollama (fast, no GPU required)
- **PDFs / DOCX / PPTX** — needs the ingestion profile (`lightonocr-vllm` on 8002, `qwen35-vllm` on 8000)

Run the cells in order. You can run the table cell alone if you only have Ollama up.

In [ ]:
from urllib.request import Request, urlopen
import json

def wipe_collection(qdrant_url=QDRANT_URL, collection=QDRANT_COLLECTION):
    """Delete the collection entirely so the next ingest starts clean."""
    base = qdrant_url.rstrip("/")
    req = Request(f"{base}/collections/{collection}", method="DELETE",
                  headers={"Content-Type": "application/json"})
    try:
        with urlopen(req, timeout=30) as r:
            result = json.loads(r.read())
        print(f"Wiped collection '{collection}': {result.get('result')}")
    except Exception as e:
        print(f"Could not delete collection (may not exist yet): {e}")

wipe_collection()

In [ ]:
import importlib
import src.ingest_table_rows as _itr_mod
importlib.reload(_itr_mod)
from src.ingest_table_rows import ingest_table_rows
import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError

INPUT_DIR = project_root / 'data' / 'input'
TABLE_EXTENSIONS = {'.xlsx', '.xlsm', '.csv'}

def get_ingested_basenames(qdrant_url=QDRANT_URL, collection=QDRANT_COLLECTION):
    """Return set of basenames already in Qdrant. Returns empty set if collection doesn't exist."""
    base = qdrant_url.rstrip("/")
    try:
        req = Request(f"{base}/collections/{collection}")
        with urlopen(req, timeout=10) as r:
            json.loads(r.read())
    except HTTPError as e:
        if e.code == 404:
            return set()
        raise

    offset = None
    names = set()
    while True:
        body = {"limit": 250, "with_payload": ["metadata.source_file"], "with_vector": False}
        if offset:
            body["offset"] = offset
        req = Request(f"{base}/collections/{collection}/points/scroll",
                      data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
        with urlopen(req) as r:
            result = json.loads(r.read())["result"]
        for p in result["points"]:
            sf = p["payload"].get("metadata", {}).get("source_file", "")
            if sf:
                names.add(sf.split("/")[-1])
        offset = result.get("next_page_offset")
        if not offset:
            break
    return names

ingested = get_ingested_basenames()
table_files = sorted(p for p in INPUT_DIR.iterdir() if p.suffix.lower() in TABLE_EXTENSIONS)

print(f"Found {len(table_files)} table file(s):")
for f in table_files:
    status = "already ingested, skipping" if f.name in ingested else "will ingest"
    print(f"  {f.name}  [{status}]")

print()
for f in table_files:
    if f.name in ingested:
        print(f"── Skipping (already ingested): {f.name}")
        continue
    print(f"── Ingesting: {f.name}")
    ingest_table_rows(str(f), collection=COLLECTION, verbose=True)
    print()

In [ ]:
import importlib
import src.ingest as _ingest_mod
importlib.reload(_ingest_mod)
from src.ingest import run_ingest

DOC_EXTENSIONS = {'.pdf', '.docx', '.doc', '.pptx', '.ppt'}

doc_files = sorted(p for p in INPUT_DIR.iterdir() if p.suffix.lower() in DOC_EXTENSIONS)
print(f"Found {len(doc_files)} document file(s):")
for f in doc_files:
    status = "already ingested, skipping" if f.name in ingested else "will ingest"
    print(f"  {f.name}  [{status}]")

print()
for f in doc_files:
    if f.name in ingested:
        print(f"── Skipping (already ingested): {f.name}")
        continue
    print(f"── Ingesting: {f.name}")
    run_ingest(
        collection=COLLECTION,
        pdf_path=f,
        qdrant_url=QDRANT_URL,
        ocr_endpoint=OCR_URL,
        ocr_model_name=OCR_MODEL,
        ollama_api_base=OLLAMA_URL,
        ollama_embed_model=EMBED_MODEL,
        enrich_with_llm=True,
        verbose=True,
    )
    print()

In [ ]:
# Verify final state
list_ingested_files()

---
## Part 3 — Retrieval (Hybrid Search)

Qdrant runs **dense + sparse (BM42) search in parallel** and fuses results with **RRF**:
```
score = 1/(rank_dense + 60) + 1/(rank_sparse + 60)
```
Dense handles semantic similarity. Sparse handles keyword/exact matching.  
Both PDF chunks and table row chunks live in the same collection — no routing needed.

### Step 0 — HyDE (Hypothetical Document Embedding)

Before embedding the query, we ask the LLM to generate a short hypothetical answer.
The hypothetical answer is written in the same style and language as the stored documents,
so its embedding matches stored chunks much better than the raw question would.

```
User query  →  LLM  →  Hypothetical answer  →  embed  →  retrieve
```

The original query is still passed to the reranker and the LLM — HyDE only affects what gets embedded.

In [ ]:
import openai, re

QUERY = 'Give me the CO2 emission for Public electricity and heat production Greece 2019'

hyde_client = openai.OpenAI(base_url=f'{GEN_URL}/v1', api_key='EMPTY')
hyde_resp = hyde_client.chat.completions.create(
    model=GEN_MODEL,
    messages=[{"role": "user", "content": (
        f"/no_think Write a short passage (2-3 sentences) that would directly answer "
        f"this question. Use the same language and terminology as the likely source document."
        f"\n\nQuestion: {QUERY}"
    )}],
    max_tokens=128,
    temperature=0.5,
)
raw = hyde_resp.choices[0].message.content
hyde_text = re.sub(r'(?s)<think>.*?</think>', '', raw).strip()

print('Original query:')
print(f'  {QUERY}')
print()
print('Hypothetical answer (this gets embedded):')
print(f'  {hyde_text}')


In [ ]:
from src.retriever import retrieve

# Use hyde_text as the embedding query (falls back to QUERY if HyDE cell wasn't run)
embed_query = hyde_text if 'hyde_text' in dir() else QUERY

results = retrieve(embed_query, top_k=20, qdrant_url=QDRANT_URL, collection=COLLECTION)

print(f'Embedded:  "{embed_query[:80]}"')
print(f'Retrieved: {len(results)} results')
print()
for i, r in enumerate(results[:8], 1):
    meta = r.get('metadata', {}) or {}
    sheet = meta.get('sheet_name', meta.get('file_name', '?'))
    print(f'[{i}] score={r["score"]:.4f} | {sheet}')
    print(f'     {r["content"][:150]}')
    print()


---
## Part 4 — Reranking

**Model:** `BAAI/bge-reranker-v2-m3` (568M, cross-encoder, multilingual)  
**How it works:** reads the query and each candidate chunk *together* and outputs a relevance score.  
Unlike embedding similarity, it understands the full context of both texts.  

Retrieval fetches top-100 candidates. Reranker picks the best 10 to send to the LLM.

In [ ]:
from src.reranker import BGEReranker

ranker = BGEReranker(model_name='BAAI/bge-reranker-v2-m3', device='cuda')

hits = retrieve(QUERY, top_k=100, qdrant_url=QDRANT_URL, collection=COLLECTION)
docs = [h['content'] for h in hits]

reranked = ranker.rerank(QUERY, docs, top_n=5)

print(f'Top 5 after reranking (from {len(hits)} candidates):')
print()
for rank, r in enumerate(reranked, 1):
    h = hits[r['index']]
    meta = h.get('metadata', {}) or {}
    print(f'[{rank}] rerank_score={r["score"]:.4f} | rrf_score={h["score"]:.4f}')
    print(f'     {h["content"][:250]}')
    print()

---
## Part 5 — Agentic RAG

**Requires query profile running** (`qwen8b-gen-vllm` on 8003)

**Why an agent?**  
A simple retrieve → generate pipeline works for single-source questions.  
The agent adds value for:
- **Multi-source questions** — searches PDFs and table rows in the same call
- **Iterative search** — can reformulate the query if the first attempt returns nothing useful  
- **Cross-document synthesis** — compare data from different files in one answer

**Model:** `Qwen/Qwen3-32B-AWQ` via vLLM with `--enable-auto-tool-choice --tool-call-parser hermes`  
**Tool:** `search_knowledge_base` — hybrid Qdrant search → BGE rerank → return top-8 chunks

In [2]:
import importlib
import src.rag_agent as _rag_mod
importlib.reload(_rag_mod)
from src.rag_agent import build_rag_agent, ask_agent

agent = build_rag_agent(
    qdrant_url=QDRANT_URL,
    collection=COLLECTION,
    retrieval_top_k=100,          # broad candidate pool
    rerank_top_n=8,               # reranker picks best 8 for LLM (context budget)
    reranker_model_name='BAAI/bge-reranker-v2-m3',
    model_name=GEN_MODEL,
    generation_api_base=GEN_URL,
)
print('Agent ready.')

/home/karvanitis/multi-modal-rag/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] Reranker 'BAAI/bge-reranker-v2-m3' loaded and warmed up.
Agent ready.


### Question 1 — PDF document question

In [3]:
answer = ask_agent(
    agent,
    "What were Greece's total greenhouse gas emissions in the base year (1990) and in 2019, and by what percentage did they change?",
    show_tool_uses=True,
)
print(answer)

2026-03-20 20:44:48.778616848 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


[TOOL_CALL] search_knowledge_base args={"query": "Greece's total greenhouse gas emissions in 1990"}
[TOOL_CALL] search_knowledge_base args={"query": "Greece's total greenhouse gas emissions in 2019"}
[TOOL_RESULT] search_knowledge_base ->
[1] file=GRC_2023_2019_11042023_103559.xlsx sheet=Table1s1 score=-0.4223
[2] file=GRC_2023_2019_11042023_103559.xlsx sheet=Table10s1 score=-0.5452
[3] file=GRC_2023_2019_11042023_103559.xlsx sheet=Table10s2 score=-0.5717
[4] file=GRC_2023_2019_11042023_103559.xlsx sheet=Table1s2 score=-0.5902
[5] file=GRC_2023_2019_11042023_103559.xlsx sheet=Table10s1 score=-0.7652
[6] file=GRC_2023_2019_11042023_103559.xlsx sheet=Table10s6 score=-0.7754
[7] file=GRC_2023_2019_11042023_103559.xlsx sheet=Table10s2 score=-0.8514
[8] file=GRC_2023_2019_11042023_103559.xlsx sheet=Table2(I)s1 score=-0.8607

[TOOL_RESULT] search_knowledge_base ->
[1] file=GRC_2023_2019_11042023_103559.xlsx sheet=Table10s1 score=3.0184
[2] file=GRC_2023_2019_11042023_103559.xlsx sheet=Table1

### Question 2 — Table question (Excel row lookup)

In [ ]:
answer = ask_agent(
    agent,
    "What were the total CO2, CH4 and N2O emissions from the Transport sector in Greece in 2019?",
    show_tool_uses=True,
)
print(answer)

### Question 3 — Cross-document question (PDF + Table)

In [ ]:
answer = ask_agent(
    agent,
    "According to the WHO compendium, what is the maximum allowable conductivity for wastewater reuse in Tunisia, and what was Greece's total SO2 emission from Energy Industries in 2019?",
    show_tool_uses=True,
)
print(answer)

---
## Utilities — Reset / Inspect Qdrant

In [ ]:
import subprocess

# List what's ingested
subprocess.run(['python', '../scripts/reset_tables.py', '--list'])

# Delete table chunks for a specific file (keep PDFs)
# subprocess.run(['python', '../scripts/reset_tables.py', '--file', 'GRC_2023_2019_11042023_103559.xlsx'])

# Delete ALL table chunks
# subprocess.run(['python', '../scripts/reset_tables.py', '--all'])